# EXAONE-Deep-7.8B QLoRA 하이퍼파라미터 최적화 실험

## 개요
RECONSTRUCTED 데이터셋 기반 체계적 하이퍼파라미터 탐색 실험 노트북

- **베이스 모델**: LGAI-EXAONE/EXAONE-Deep-7.8B
- **베이스라인 (EXP-001)**: r=16, lr=2e-4, 1 epoch → BLEU=17.32, ROUGE-L=18.28
- **목표**: BLEU >= 30, ROUGE-L >= 40
- **환경**: Google Colab A100 (40GB VRAM)

## 실험 라운드 구성
| 라운드 | 목적 | 고정 변수 | 탐색 변수 |
|--------|------|-----------|----------|
| Round 1 | Epoch + Learning Rate | r=32, alpha=64 | lr, epochs, scheduler |
| Round 2 | LoRA Rank | best epoch/lr from R1 | r, alpha |
| Round 3 | 학습 기법 비교 | best config from R1+R2 | CompletionOnly, thought tags |

## 1. 환경 설정 및 패키지 설치

In [ ]:
# === Colab 환경 셋업 ===
import os

# 1. 프로젝트 클론
if not os.path.exists("/content/GovOn"):
    !git clone https://github.com/GovOn-Org/GovOn.git /content/GovOn

# 2. 작업 디렉토리 설정
os.chdir("/content/GovOn")
PROJECT_ROOT = "/content/GovOn"

# 3. 패키지 설치
!pip install -q transformers peft trl bitsandbytes datasets wandb sacrebleu rouge_score bert_score accelerate tqdm

# 4. 경로 설정
TRAIN_PATH = f"{PROJECT_ROOT}/data/processed/civil_complaint_train.jsonl"
VAL_PATH = f"{PROJECT_ROOT}/data/processed/civil_complaint_val.jsonl"
TEST_PATH = f"{PROJECT_ROOT}/data/processed/civil_complaint_test.jsonl"
OUTPUT_BASE = f"{PROJECT_ROOT}/models/checkpoints"
RESULTS_DIR = f"{PROJECT_ROOT}/docs/outputs/experiments"
os.makedirs(OUTPUT_BASE, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)

# 5. GPU 확인 (A100 필수)
import torch
gpu_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "None"
gpu_mem = torch.cuda.get_device_properties(0).total_mem / 1e9 if torch.cuda.is_available() else 0
print(f"GPU: {gpu_name}, VRAM: {gpu_mem:.1f} GB")
assert torch.cuda.is_available(), "GPU가 필요합니다!"

# 6. 데이터 파일 존재 확인
for p in [TRAIN_PATH, VAL_PATH, TEST_PATH]:
    assert os.path.exists(p), f"파일을 찾을 수 없습니다: {p}"
    line_count = sum(1 for _ in open(p))
    print(f"  {os.path.basename(p)}: {line_count:,} samples")

# 7. W&B 로그인
import wandb
wandb.login()

In [ ]:
# ============================================================
# 1-5. 공통 임포트 및 상수 정의
# ============================================================
import gc
import json
import re
import random
import numpy as np
from datetime import datetime
from tqdm.auto import tqdm

from datasets import load_dataset
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    TrainingArguments,
    set_seed,
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer, DataCollatorForCompletionOnlyLM

import sacrebleu
from rouge_score import rouge_scorer
from bert_score import score as bert_score_fn

# 재현성을 위한 시드
SEED = 42
set_seed(SEED)
random.seed(SEED)

# 모델 설정
BASE_MODEL_ID = "LGAI-EXAONE/EXAONE-Deep-7.8B"
MAX_SEQ_LENGTH = 2048

# 베이스라인 지표 (EXP-001)
BASELINE = {"bleu": 17.32, "rouge_l": 18.28}
TARGETS  = {"bleu": 30.0, "rouge_l": 40.0}

# 시스템 프롬프트
SYSTEM_PROMPT = "당신은 지자체 민원 담당 공무원을 돕는 AI 어시스턴트입니다."

print("모든 임포트 완료.")

## 2. 실험 설정 (Experiment Configuration)

3단계 라운드 방식으로 진행합니다:
- **Round 1**: Epoch + Learning Rate 탐색 (LoRA rank 고정: r=32)
- **Round 2**: LoRA Rank 탐색 (R1 최적 epoch/lr 사용)
- **Round 3**: 학습 기법 비교 (CompletionOnly, thought 태그)

In [ ]:
# ============================================================
# 2. 실험 설정 정의
# ============================================================

# --- Round 1: Epoch + Learning Rate 탐색 (r=32, alpha=64 고정) ---
ROUND1_EXPERIMENTS = [
    {
        "exp_id": "R1-A",
        "description": "lr=1e-4, 2 epochs, cosine",
        "lora_r": 32,
        "lora_alpha": 64,
        "lr": 1e-4,
        "epochs": 2,
        "scheduler": "cosine",
        "lora_dropout": 0.10,
        "warmup_ratio": 0.05,
    },
    {
        "exp_id": "R1-B",
        "description": "lr=1e-4, 3 epochs, cosine",
        "lora_r": 32,
        "lora_alpha": 64,
        "lr": 1e-4,
        "epochs": 3,
        "scheduler": "cosine",
        "lora_dropout": 0.10,
        "warmup_ratio": 0.05,
    },
    {
        "exp_id": "R1-C",
        "description": "lr=5e-5, 3 epochs, cosine",
        "lora_r": 32,
        "lora_alpha": 64,
        "lr": 5e-5,
        "epochs": 3,
        "scheduler": "cosine",
        "lora_dropout": 0.10,
        "warmup_ratio": 0.05,
    },
]

# --- Round 2: LoRA Rank 탐색 (R1 최적 epoch/lr 적용) ---
# 아래 값은 R1 결과를 보고 업데이트합니다.
ROUND2_EXPERIMENTS = [
    {
        "exp_id": "R2-A",
        "description": "LoRA r=16, alpha=32",
        "lora_r": 16,
        "lora_alpha": 32,
        "lora_dropout": 0.10,
    },
    {
        "exp_id": "R2-B",
        "description": "LoRA r=32, alpha=64",
        "lora_r": 32,
        "lora_alpha": 64,
        "lora_dropout": 0.10,
    },
    {
        "exp_id": "R2-C",
        "description": "LoRA r=64, alpha=128",
        "lora_r": 64,
        "lora_alpha": 128,
        "lora_dropout": 0.10,
    },
]

# --- Round 3: 학습 기법 비교 (R1+R2 최적 설정 적용) ---
ROUND3_EXPERIMENTS = [
    {
        "exp_id": "R3-A",
        "description": "표준 SFT + thought 태그 포함",
        "use_completion_only": False,
        "with_thought": True,
    },
    {
        "exp_id": "R3-B",
        "description": "CompletionOnly + thought 태그 제거",
        "use_completion_only": True,
        "with_thought": False,
    },
]

print(f"Round 1: {len(ROUND1_EXPERIMENTS)} 실험 (Epoch + LR)")
print(f"Round 2: {len(ROUND2_EXPERIMENTS)} 실험 (LoRA Rank)")
print(f"Round 3: {len(ROUND3_EXPERIMENTS)} 실험 (학습 기법)")
print(f"총 {len(ROUND1_EXPERIMENTS) + len(ROUND2_EXPERIMENTS) + len(ROUND3_EXPERIMENTS)} 실험")

## 3. 데이터 로딩 및 포맷팅 함수

In [ ]:
# ============================================================
# 3-1. 데이터 로딩 함수
# ============================================================

def load_jsonl(path, max_samples=None):
    """JSONL 파일을 로드합니다."""
    data = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                try:
                    data.append(json.loads(line))
                except json.JSONDecodeError:
                    continue
    if max_samples:
        random.seed(SEED)
        random.shuffle(data)
        data = data[:max_samples]
    return data


def remove_thought_tags(text):
    """<thought>...</thought> 태그를 제거합니다."""
    return re.sub(r"<thought>.*?</thought>", "", text, flags=re.DOTALL).strip()


def normalize_pii(text):
    """PII 마스킹 토큰을 정규화합니다."""
    text = re.sub(r"\[NAME_MASKED\]", "[NAME]", text)
    text = re.sub(r"\[PHONE_MASKED\]", "[PHONE]", text)
    text = re.sub(r"\[ADDRESS_MASKED\]", "[ADDRESS]", text)
    return text


# 평가 데이터 미리 로드
eval_data = load_jsonl(TEST_PATH, max_samples=100)
print(f"평가 데이터: {len(eval_data)} 샘플 로드됨")
print(f"샘플 키: {list(eval_data[0].keys())}")
print(f"샘플 instruction: {eval_data[0]['instruction'][:80]}...")

In [ ]:
# ============================================================
# 3-2. 포맷팅 함수 (EXAONE Chat Template)
# ============================================================

def create_formatting_func(tokenizer, with_thought=True):
    """
    EXAONE 채팅 템플릿 기반 포맷팅 함수를 생성합니다.
    
    Args:
        tokenizer: EXAONE 토크나이저
        with_thought: True이면 <thought> 태그를 유지, False이면 제거
    """
    def formatting_prompts_func(example):
        output_texts = []
        for i in range(len(example["instruction"])):
            output_text = example["output"][i]
            if not with_thought:
                output_text = remove_thought_tags(output_text)
            
            messages = [
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": f"{example['instruction'][i]}\n\n{example['input'][i]}"},
                {"role": "assistant", "content": output_text},
            ]
            text = tokenizer.apply_chat_template(
                messages, tokenize=False, add_generation_prompt=False
            )
            output_texts.append(text)
        return output_texts
    
    return formatting_prompts_func


print("포맷팅 함수 정의 완료.")

## 4. 모델 로딩 함수

EXAONE-Deep-7.8B를 QLoRA (4-bit NF4)로 로드하고, EXAONE 호환성 패치를 적용합니다.

In [ ]:
# ============================================================
# 4. 모델 로딩 함수
# ============================================================

def load_model_with_qlora(model_id, lora_r, lora_alpha, lora_dropout=0.10):
    """
    EXAONE-Deep-7.8B를 QLoRA 4-bit으로 로드합니다.
    
    Args:
        model_id: HuggingFace 모델 ID
        lora_r: LoRA rank
        lora_alpha: LoRA alpha
        lora_dropout: LoRA dropout (기본값 0.10 - 베이스라인 대비 증가)
    
    Returns:
        model, tokenizer
    """
    print(f"\n모델 로드 중: {model_id}")
    print(f"  LoRA config: r={lora_r}, alpha={lora_alpha}, dropout={lora_dropout}")
    
    # 토크나이저
    tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)
    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.padding_side = "right"
    
    # 4-bit 양자화 설정
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.bfloat16,
        bnb_4bit_use_double_quant=True,
    )
    
    # 모델 로드
    model = AutoModelForCausalLM.from_pretrained(
        model_id,
        quantization_config=bnb_config,
        device_map="auto",
        trust_remote_code=True,
        torch_dtype=torch.bfloat16,
    )
    
    # EXAONE 호환성 패치 (transformers 버전에 따라 필요)
    try:
        model.get_input_embeddings()
    except (NotImplementedError, AttributeError):
        model.get_input_embeddings = lambda: model.transformer.wte
        print("  [패치] get_input_embeddings")
    
    try:
        model.get_output_embeddings()
    except (NotImplementedError, AttributeError):
        model.get_output_embeddings = lambda: model.lm_head
        print("  [패치] get_output_embeddings")
    
    # kbit 학습 준비
    model = prepare_model_for_kbit_training(model)
    
    # LoRA 적용
    lora_config = LoraConfig(
        r=lora_r,
        lora_alpha=lora_alpha,
        target_modules=[
            "q_proj", "k_proj", "v_proj", "o_proj",
            "gate_proj", "up_proj", "down_proj",
        ],
        lora_dropout=lora_dropout,
        bias="none",
        task_type="CAUSAL_LM",
    )
    model = get_peft_model(model, lora_config)
    model.print_trainable_parameters()
    
    return model, tokenizer


def cleanup_model(model):
    """모델을 메모리에서 해제합니다."""
    del model
    gc.collect()
    torch.cuda.empty_cache()
    # 메모리 상태 출력
    if torch.cuda.is_available():
        allocated = torch.cuda.memory_allocated() / 1024**3
        reserved = torch.cuda.memory_reserved() / 1024**3
        print(f"  GPU 메모리: {allocated:.1f}GB allocated, {reserved:.1f}GB reserved")


print("모델 로딩 함수 정의 완료.")

## 5. 학습 함수 (SFTTrainer)

In [ ]:
# ============================================================
# 5. 학습 함수
# ============================================================

def train_experiment(
    model,
    tokenizer,
    exp_id,
    lr,
    epochs,
    scheduler="cosine",
    warmup_ratio=0.05,
    batch_size=2,
    grad_accum=8,
    use_completion_only=False,
    with_thought=True,
    wandb_group="default",
    wandb_tags=None,
):
    """
    단일 실험을 학습합니다.
    
    Args:
        model: QLoRA 적용된 모델
        tokenizer: 토크나이저
        exp_id: 실험 ID
        lr: 학습률
        epochs: 에폭 수
        scheduler: 스케줄러 타입 (cosine, linear)
        warmup_ratio: 워밍업 비율 (베이스라인 0.03 -> 0.05로 증가)
        batch_size: 배치 크기 (A100 최적화: 2-4)
        grad_accum: 그래디언트 누적 (effective batch = batch_size * grad_accum)
        use_completion_only: DataCollatorForCompletionOnlyLM 사용 여부
        with_thought: <thought> 태그 포함 여부
        wandb_group: W&B 그룹명
        wandb_tags: W&B 태그 리스트
    
    Returns:
        trainer: 학습 완료된 Trainer
    """
    output_dir = os.path.join(OUTPUT_BASE, exp_id)
    os.makedirs(output_dir, exist_ok=True)
    
    effective_batch = batch_size * grad_accum
    print(f"\n{'='*65}")
    print(f"[학습 시작] {exp_id}")
    print(f"  lr={lr}, epochs={epochs}, scheduler={scheduler}")
    print(f"  batch={batch_size} x grad_accum={grad_accum} = effective {effective_batch}")
    print(f"  warmup_ratio={warmup_ratio}, completion_only={use_completion_only}")
    print(f"  with_thought={with_thought}")
    print(f"{'='*65}")
    
    # W&B run 초기화
    run = wandb.init(
        project=WANDB_PROJECT,
        name=exp_id,
        group=wandb_group,
        tags=wandb_tags or ["qlora", "exaone", "hparam-search"],
        config={
            "exp_id": exp_id,
            "base_model": BASE_MODEL_ID,
            "learning_rate": lr,
            "num_epochs": epochs,
            "lr_scheduler": scheduler,
            "warmup_ratio": warmup_ratio,
            "batch_size": batch_size,
            "grad_accum": grad_accum,
            "effective_batch_size": effective_batch,
            "use_completion_only": use_completion_only,
            "with_thought": with_thought,
            "max_seq_length": MAX_SEQ_LENGTH,
            "seed": SEED,
            "baseline_bleu": BASELINE["bleu"],
            "baseline_rouge_l": BASELINE["rouge_l"],
        },
        reinit=True,
    )
    
    # 데이터셋 로드
    dataset = load_dataset(
        "json",
        data_files={"train": TRAIN_PATH, "validation": VAL_PATH},
    )
    
    # 포맷팅 함수
    formatting_func = create_formatting_func(tokenizer, with_thought=with_thought)
    
    # TrainingArguments (A100 최적화)
    training_args = TrainingArguments(
        output_dir=output_dir,
        per_device_train_batch_size=batch_size,
        per_device_eval_batch_size=batch_size,
        gradient_accumulation_steps=grad_accum,
        eval_accumulation_steps=10,
        learning_rate=lr,
        num_train_epochs=epochs,
        lr_scheduler_type=scheduler,
        warmup_ratio=warmup_ratio,
        weight_decay=0.01,
        max_grad_norm=1.0,
        bf16=True,
        tf32=True,
        logging_steps=10,
        eval_strategy="steps",
        eval_steps=100,
        save_strategy="steps",
        save_steps=100,
        save_total_limit=2,
        load_best_model_at_end=True,
        metric_for_best_model="eval_loss",
        greater_is_better=False,
        report_to="wandb",
        run_name=exp_id,
        gradient_checkpointing=True,
        optim="paged_adamw_8bit",
        seed=SEED,
        dataloader_num_workers=2,
    )
    
    # DataCollator 설정
    data_collator = None
    if use_completion_only:
        # assistant 응답 부분만 loss 계산
        # EXAONE 포맷에서 assistant 시작 토큰을 response_template으로 사용
        response_template = "[|assistant|]"
        data_collator = DataCollatorForCompletionOnlyLM(
            response_template=response_template,
            tokenizer=tokenizer,
        )
        print(f"  DataCollatorForCompletionOnlyLM 활성화 (response_template='{response_template}')")
    
    # SFTTrainer
    trainer_kwargs = dict(
        model=model,
        train_dataset=dataset["train"],
        eval_dataset=dataset["validation"],
        max_seq_length=MAX_SEQ_LENGTH,
        tokenizer=tokenizer,
        formatting_func=formatting_func,
        args=training_args,
    )
    if data_collator is not None:
        trainer_kwargs["data_collator"] = data_collator
    
    trainer = SFTTrainer(**trainer_kwargs)
    
    # 학습 실행
    train_result = trainer.train()
    
    # 최종 모델 저장
    final_dir = os.path.join(output_dir, "final")
    trainer.save_model(final_dir)
    tokenizer.save_pretrained(final_dir)
    print(f"  모델 저장 완료: {final_dir}")
    
    # 학습 지표 로깅
    train_metrics = train_result.metrics
    wandb.log({f"train/{k}": v for k, v in train_metrics.items()})
    print(f"  학습 loss: {train_metrics.get('train_loss', 'N/A'):.4f}")
    
    return trainer, run


print("학습 함수 정의 완료.")

## 6. 평가 함수 (SacreBLEU + ROUGE-L + BERTScore)

**베이스라인 대비 개선 사항:**
- SacreBLEU 사용 (unigram 근사 대신 표준 BLEU)
- 시스템 메시지를 평가 프롬프트에 포함
- `<thought>` 태그 제거 후 메트릭 계산
- PII 토큰 정규화
- BERTScore F1 추가

In [ ]:
# ============================================================
# 6. 평가 함수 (SacreBLEU, ROUGE-L, BERTScore)
# ============================================================

def evaluate_model(
    model,
    tokenizer,
    eval_samples,
    max_new_tokens=150,
    temperature=0.3,
    top_p=0.9,
    repetition_penalty=1.3,
    compute_bertscore=True,
):
    """
    모델을 평가하고 SacreBLEU, ROUGE-L, BERTScore를 계산합니다.
    
    Args:
        model: 평가할 모델
        tokenizer: 토크나이저
        eval_samples: 평가 데이터 (list of dict)
        max_new_tokens: 최대 생성 토큰 수
        temperature: 샘플링 온도
        top_p: nucleus sampling 확률
        repetition_penalty: 반복 페널티
        compute_bertscore: BERTScore 계산 여부 (느릴 수 있음)
    
    Returns:
        dict: {bleu, rouge_l, bertscore_f1, predictions, references}
    """
    model.eval()
    
    predictions = []
    references = []
    generation_errors = 0
    
    print(f"  평가 시작: {len(eval_samples)} 샘플")
    
    for item in tqdm(eval_samples, desc="평가 중"):
        try:
            # 시스템 메시지를 포함한 프롬프트 구성 (베이스라인 대비 개선)
            messages = [
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": f"{item['instruction']}\n\n{item['input']}"},
            ]
            
            input_ids = tokenizer.apply_chat_template(
                messages,
                tokenize=True,
                add_generation_prompt=True,
                return_tensors="pt",
            ).to(model.device)
            
            # 입력이 텐서인지 dict인지 확인
            if isinstance(input_ids, dict):
                gen_input = input_ids
                input_len = gen_input["input_ids"].shape[1]
            else:
                gen_input = {"input_ids": input_ids}
                input_len = input_ids.shape[1]
            
            with torch.no_grad():
                output_ids = model.generate(
                    **gen_input,
                    max_new_tokens=max_new_tokens,
                    do_sample=True,
                    temperature=temperature,
                    top_p=top_p,
                    repetition_penalty=repetition_penalty,
                    eos_token_id=tokenizer.eos_token_id,
                )
            
            # 생성된 토큰만 추출
            generated_ids = output_ids[0][input_len:]
            generated_text = tokenizer.decode(generated_ids, skip_special_tokens=True).strip()
            
            # 후처리: <thought> 태그 제거 + PII 정규화
            generated_clean = normalize_pii(remove_thought_tags(generated_text))
            reference_clean = normalize_pii(remove_thought_tags(item.get("output", "")))
            
            if generated_clean and reference_clean:
                predictions.append(generated_clean)
                references.append(reference_clean)
            else:
                generation_errors += 1
                
        except Exception as e:
            generation_errors += 1
            if generation_errors <= 3:
                print(f"  [경고] 생성 오류: {str(e)[:100]}")
            continue
    
    if not predictions:
        print("  [오류] 유효한 예측이 없습니다.")
        return {"bleu": 0.0, "rouge_l": 0.0, "bertscore_f1": 0.0,
                "predictions": [], "references": [], "errors": generation_errors}
    
    # --- SacreBLEU (표준 BLEU) ---
    bleu_result = sacrebleu.corpus_bleu(
        predictions,
        [references],  # sacrebleu는 references를 리스트의 리스트로 받음
    )
    bleu_score = bleu_result.score
    
    # --- ROUGE-L ---
    rouge = rouge_scorer.RougeScorer(["rougeL"], use_stemmer=False)
    rouge_scores = []
    for pred, ref in zip(predictions, references):
        score = rouge.score(ref, pred)
        rouge_scores.append(score["rougeL"].fmeasure * 100)
    rouge_l_score = float(np.mean(rouge_scores))
    
    # --- BERTScore ---
    bertscore_f1 = 0.0
    if compute_bertscore and len(predictions) > 0:
        try:
            P, R, F1 = bert_score_fn(
                predictions,
                references,
                lang="ko",
                verbose=False,
            )
            bertscore_f1 = float(F1.mean()) * 100
        except Exception as e:
            print(f"  [경고] BERTScore 계산 실패: {e}")
    
    # 결과 출력
    print(f"\n  === 평가 결과 ===")
    print(f"  유효 샘플: {len(predictions)}/{len(eval_samples)} (오류: {generation_errors})")
    print(f"  SacreBLEU:    {bleu_score:.2f}  (베이스라인: {BASELINE['bleu']}, 목표: >={TARGETS['bleu']})")
    print(f"  ROUGE-L:      {rouge_l_score:.2f}  (베이스라인: {BASELINE['rouge_l']}, 목표: >={TARGETS['rouge_l']})")
    print(f"  BERTScore F1: {bertscore_f1:.2f}")
    
    bleu_met = bleu_score >= TARGETS["bleu"]
    rouge_met = rouge_l_score >= TARGETS["rouge_l"]
    print(f"  BLEU 목표 달성:   {'PASS' if bleu_met else 'FAIL'}")
    print(f"  ROUGE-L 목표 달성: {'PASS' if rouge_met else 'FAIL'}")
    
    return {
        "bleu": bleu_score,
        "rouge_l": rouge_l_score,
        "bertscore_f1": bertscore_f1,
        "bleu_detail": str(bleu_result),
        "num_valid": len(predictions),
        "num_errors": generation_errors,
        "predictions": predictions,
        "references": references,
    }


print("평가 함수 정의 완료.")

## 7. 실험 실행 헬퍼 함수

각 라운드의 실험을 순차 실행하고, 메모리를 정리하며, 결과를 수집합니다.

In [ ]:
# ============================================================
# 7. 실험 실행 헬퍼
# ============================================================

# 전체 결과 저장소
ALL_RESULTS = []


def run_experiment(
    exp_config,
    wandb_group,
    eval_samples,
    wandb_tags=None,
):
    """
    단일 실험을 실행합니다 (모델 로드 -> 학습 -> 평가 -> 메모리 정리).
    
    Args:
        exp_config: 실험 설정 dict
        wandb_group: W&B 그룹명
        eval_samples: 평가 데이터
        wandb_tags: W&B 태그 리스트
    
    Returns:
        dict: 실험 결과
    """
    exp_id = exp_config["exp_id"]
    result = {"exp_id": exp_id, "config": exp_config}
    
    try:
        # 1. 모델 로드
        model, tokenizer = load_model_with_qlora(
            model_id=BASE_MODEL_ID,
            lora_r=exp_config["lora_r"],
            lora_alpha=exp_config["lora_alpha"],
            lora_dropout=exp_config.get("lora_dropout", 0.10),
        )
        
        # 2. 학습
        trainer, run = train_experiment(
            model=model,
            tokenizer=tokenizer,
            exp_id=exp_id,
            lr=exp_config["lr"],
            epochs=exp_config["epochs"],
            scheduler=exp_config.get("scheduler", "cosine"),
            warmup_ratio=exp_config.get("warmup_ratio", 0.05),
            use_completion_only=exp_config.get("use_completion_only", False),
            with_thought=exp_config.get("with_thought", True),
            wandb_group=wandb_group,
            wandb_tags=wandb_tags,
        )
        
        # 3. 평가
        print(f"\n[평가] {exp_id}")
        eval_result = evaluate_model(
            model=model,
            tokenizer=tokenizer,
            eval_samples=eval_samples,
        )
        
        # 4. W&B 로깅
        wandb.log({
            "eval/sacrebleu": eval_result["bleu"],
            "eval/rouge_l": eval_result["rouge_l"],
            "eval/bertscore_f1": eval_result["bertscore_f1"],
            "eval/bleu_improvement": eval_result["bleu"] - BASELINE["bleu"],
            "eval/rouge_improvement": eval_result["rouge_l"] - BASELINE["rouge_l"],
            "eval/bleu_target_met": int(eval_result["bleu"] >= TARGETS["bleu"]),
            "eval/rouge_target_met": int(eval_result["rouge_l"] >= TARGETS["rouge_l"]),
        })
        
        result.update({
            "bleu": eval_result["bleu"],
            "rouge_l": eval_result["rouge_l"],
            "bertscore_f1": eval_result["bertscore_f1"],
            "bleu_detail": eval_result["bleu_detail"],
            "num_valid": eval_result["num_valid"],
            "num_errors": eval_result["num_errors"],
            "bleu_target_met": eval_result["bleu"] >= TARGETS["bleu"],
            "rouge_target_met": eval_result["rouge_l"] >= TARGETS["rouge_l"],
            "wandb_run_id": run.id,
            "wandb_run_url": run.url,
        })
        
    except Exception as e:
        print(f"\n[오류] 실험 {exp_id} 실패: {e}")
        import traceback
        traceback.print_exc()
        result["error"] = str(e)
    
    finally:
        # 5. 메모리 정리
        print(f"\n[정리] {exp_id} 메모리 해제 중...")
        try:
            cleanup_model(model)
            del trainer
        except NameError:
            pass
        gc.collect()
        torch.cuda.empty_cache()
        wandb.finish()
    
    ALL_RESULTS.append(result)
    
    # 중간 결과 저장
    intermediate_path = os.path.join(RESULTS_DIR, "hparam_search_intermediate.json")
    with open(intermediate_path, "w", encoding="utf-8") as f:
        json.dump(ALL_RESULTS, f, indent=2, ensure_ascii=False, default=str)
    
    return result


def find_best_result(results, metric="combined"):
    """
    결과 목록에서 최적 설정을 선택합니다.
    
    Args:
        results: 실험 결과 리스트
        metric: 'bleu', 'rouge_l', 'combined' (BLEU + ROUGE-L)
    """
    valid = [r for r in results if "bleu" in r and "error" not in r]
    if not valid:
        print("유효한 결과가 없습니다.")
        return None
    
    if metric == "combined":
        valid.sort(key=lambda x: x["bleu"] + x["rouge_l"], reverse=True)
    else:
        valid.sort(key=lambda x: x[metric], reverse=True)
    
    best = valid[0]
    print(f"\n  최적 실험: {best['exp_id']}")
    print(f"  BLEU: {best['bleu']:.2f}, ROUGE-L: {best['rouge_l']:.2f}, BERTScore: {best['bertscore_f1']:.2f}")
    return best


def print_results_table(results, title="실험 결과"):
    """결과 비교 테이블을 출력합니다."""
    print(f"\n{'='*85}")
    print(f"  {title}")
    print(f"{'='*85}")
    print(f"{'Exp ID':<10} {'BLEU':>8} {'ROUGE-L':>9} {'BERTScore':>10} {'d BLEU':>8} {'d ROUGE':>8} {'상태':<10}")
    print(f"{'-'*85}")
    
    for r in results:
        if "error" in r:
            print(f"{r['exp_id']:<10} {'ERROR':>8} {'-':>9} {'-':>10} {'-':>8} {'-':>8} {'FAILED':<10}")
            continue
        if "bleu" not in r:
            continue
        
        d_bleu = r["bleu"] - BASELINE["bleu"]
        d_rouge = r["rouge_l"] - BASELINE["rouge_l"]
        bleu_ok = r["bleu"] >= TARGETS["bleu"]
        rouge_ok = r["rouge_l"] >= TARGETS["rouge_l"]
        
        if bleu_ok and rouge_ok:
            status = "BOTH PASS"
        elif bleu_ok or rouge_ok:
            status = "PARTIAL"
        else:
            status = "BELOW"
        
        print(
            f"{r['exp_id']:<10} "
            f"{r['bleu']:>8.2f} "
            f"{r['rouge_l']:>9.2f} "
            f"{r['bertscore_f1']:>10.2f} "
            f"{d_bleu:>+8.2f} "
            f"{d_rouge:>+8.2f} "
            f"{status:<10}"
        )
    
    print(f"{'-'*85}")
    print(f"{'Baseline':<10} {BASELINE['bleu']:>8.2f} {BASELINE['rouge_l']:>9.2f} {'-':>10} {'-':>8} {'-':>8} {'reference':<10}")
    print(f"{'Target':<10} {TARGETS['bleu']:>8.2f} {TARGETS['rouge_l']:>9.2f} {'-':>10} {'-':>8} {'-':>8} {'-':<10}")


print("실험 헬퍼 함수 정의 완료.")

---
## 8. Round 1 실행: Epoch + Learning Rate 탐색

**고정**: r=32, alpha=64, dropout=0.10  
**탐색**: lr (1e-4, 5e-5), epochs (2, 3), scheduler (cosine)

| 실험 | LR | Epochs | 예상 효과 |
|------|-----|--------|----------|
| R1-A | 1e-4 | 2 | 안정적 수렴 |
| R1-B | 1e-4 | 3 | 추가 학습 효과 확인 |
| R1-C | 5e-5 | 3 | 보수적 lr + 충분한 에폭 |

In [ ]:
# ============================================================
# 8. Round 1 실행
# ============================================================

print("="*65)
print("  ROUND 1: Epoch + Learning Rate 탐색")
print("  고정: r=32, alpha=64, dropout=0.10")
print("="*65)

round1_results = []

for exp_cfg in ROUND1_EXPERIMENTS:
    result = run_experiment(
        exp_config=exp_cfg,
        wandb_group="Round1-Epoch-LR",
        eval_samples=eval_data,
        wandb_tags=["qlora", "exaone", "round1", "epoch-lr-search"],
    )
    round1_results.append(result)

# Round 1 결과 요약
print_results_table(round1_results, title="Round 1 결과: Epoch + Learning Rate")
best_r1 = find_best_result(round1_results)

In [ ]:
# ============================================================
# 8-2. Round 1 최적 설정 확인 및 Round 2에 적용
# ============================================================

if best_r1 is None:
    print("[오류] Round 1에서 유효한 결과가 없습니다. 기본값을 사용합니다.")
    BEST_LR = 1e-4
    BEST_EPOCHS = 3
    BEST_SCHEDULER = "cosine"
    BEST_WARMUP = 0.05
else:
    BEST_LR = best_r1["config"]["lr"]
    BEST_EPOCHS = best_r1["config"]["epochs"]
    BEST_SCHEDULER = best_r1["config"].get("scheduler", "cosine")
    BEST_WARMUP = best_r1["config"].get("warmup_ratio", 0.05)

print(f"\nRound 2에 적용할 설정:")
print(f"  LR: {BEST_LR}")
print(f"  Epochs: {BEST_EPOCHS}")
print(f"  Scheduler: {BEST_SCHEDULER}")
print(f"  Warmup: {BEST_WARMUP}")

# Round 2 실험에 최적 LR/Epoch 적용
for exp in ROUND2_EXPERIMENTS:
    exp["lr"] = BEST_LR
    exp["epochs"] = BEST_EPOCHS
    exp["scheduler"] = BEST_SCHEDULER
    exp["warmup_ratio"] = BEST_WARMUP

print("\nRound 2 실험 설정 업데이트 완료.")
for exp in ROUND2_EXPERIMENTS:
    print(f"  {exp['exp_id']}: r={exp['lora_r']}, alpha={exp['lora_alpha']}, lr={exp['lr']}, epochs={exp['epochs']}")

---
## 9. Round 2 실행: LoRA Rank 탐색

**고정**: Round 1 최적 lr, epochs  
**탐색**: r (16, 32, 64), alpha (2*r)

| 실험 | Rank | Alpha | 파라미터 규모 |
|------|------|-------|--------------|
| R2-A | 16 | 32 | ~24M |
| R2-B | 32 | 64 | ~48M |
| R2-C | 64 | 128 | ~96M |

In [ ]:
# ============================================================
# 9. Round 2 실행
# ============================================================

print("="*65)
print("  ROUND 2: LoRA Rank 탐색")
print(f"  고정: lr={BEST_LR}, epochs={BEST_EPOCHS}")
print("="*65)

round2_results = []

for exp_cfg in ROUND2_EXPERIMENTS:
    result = run_experiment(
        exp_config=exp_cfg,
        wandb_group="Round2-LoRA-Rank",
        eval_samples=eval_data,
        wandb_tags=["qlora", "exaone", "round2", "rank-search"],
    )
    round2_results.append(result)

# Round 2 결과 요약
print_results_table(round2_results, title="Round 2 결과: LoRA Rank")
best_r2 = find_best_result(round2_results)

In [ ]:
# ============================================================
# 9-2. Round 2 최적 설정 확인 및 Round 3에 적용
# ============================================================

if best_r2 is None:
    print("[오류] Round 2에서 유효한 결과가 없습니다. 기본값을 사용합니다.")
    BEST_LORA_R = 32
    BEST_LORA_ALPHA = 64
else:
    BEST_LORA_R = best_r2["config"]["lora_r"]
    BEST_LORA_ALPHA = best_r2["config"]["lora_alpha"]

print(f"\nRound 3에 적용할 전체 설정:")
print(f"  LoRA r: {BEST_LORA_R}")
print(f"  LoRA alpha: {BEST_LORA_ALPHA}")
print(f"  LR: {BEST_LR}")
print(f"  Epochs: {BEST_EPOCHS}")
print(f"  Scheduler: {BEST_SCHEDULER}")

# Round 3 실험에 최적 설정 적용
for exp in ROUND3_EXPERIMENTS:
    exp["lora_r"] = BEST_LORA_R
    exp["lora_alpha"] = BEST_LORA_ALPHA
    exp["lr"] = BEST_LR
    exp["epochs"] = BEST_EPOCHS
    exp["scheduler"] = BEST_SCHEDULER
    exp["warmup_ratio"] = BEST_WARMUP
    exp["lora_dropout"] = 0.10

print("\nRound 3 실험 설정 업데이트 완료.")
for exp in ROUND3_EXPERIMENTS:
    print(f"  {exp['exp_id']}: completion_only={exp['use_completion_only']}, with_thought={exp['with_thought']}")

---
## 10. Round 3 실행: 학습 기법 비교

**고정**: Round 1+2 최적 설정 (lr, epochs, rank, alpha)  
**탐색**: 학습 방식

| 실험 | CompletionOnly | Thought 태그 | 설명 |
|------|---------------|-------------|------|
| R3-A | No | Yes | 표준 SFT + thought 포함 (전체 시퀀스 loss) |
| R3-B | Yes | No | assistant 응답만 loss + thought 제거 |

In [ ]:
# ============================================================
# 10. Round 3 실행
# ============================================================

print("="*65)
print("  ROUND 3: 학습 기법 비교")
print(f"  고정: r={BEST_LORA_R}, lr={BEST_LR}, epochs={BEST_EPOCHS}")
print("="*65)

round3_results = []

for exp_cfg in ROUND3_EXPERIMENTS:
    result = run_experiment(
        exp_config=exp_cfg,
        wandb_group="Round3-Training-Technique",
        eval_samples=eval_data,
        wandb_tags=["qlora", "exaone", "round3", "technique-comparison"],
    )
    round3_results.append(result)

# Round 3 결과 요약
print_results_table(round3_results, title="Round 3 결과: 학습 기법 비교")
best_r3 = find_best_result(round3_results)

---
## 11. 전체 결과 요약 및 최종 모델 선택

In [ ]:
# ============================================================
# 11-1. 전체 결과 요약
# ============================================================

print("\n" + "#"*85)
print("#  전체 하이퍼파라미터 최적화 결과 요약")
print("#"*85)

# 라운드별 결과 출력
print_results_table(round1_results, title="Round 1: Epoch + Learning Rate")
print_results_table(round2_results, title="Round 2: LoRA Rank")
print_results_table(round3_results, title="Round 3: 학습 기법")

# 전체 최적 모델 선택
print("\n" + "="*85)
print("  전체 최적 모델 선택")
print("="*85)

overall_best = find_best_result(ALL_RESULTS)

if overall_best:
    cfg = overall_best["config"]
    print(f"\n  === 최종 추천 설정 ===")
    print(f"  실험 ID:     {overall_best['exp_id']}")
    print(f"  LoRA r:      {cfg.get('lora_r', 'N/A')}")
    print(f"  LoRA alpha:  {cfg.get('lora_alpha', 'N/A')}")
    print(f"  LR:          {cfg.get('lr', 'N/A')}")
    print(f"  Epochs:      {cfg.get('epochs', 'N/A')}")
    print(f"  Scheduler:   {cfg.get('scheduler', 'N/A')}")
    print(f"  Dropout:     {cfg.get('lora_dropout', 'N/A')}")
    print(f"  Warmup:      {cfg.get('warmup_ratio', 'N/A')}")
    print(f"  CompletionOnly: {cfg.get('use_completion_only', False)}")
    print(f"  With Thought:   {cfg.get('with_thought', True)}")
    print(f"")
    print(f"  SacreBLEU:    {overall_best['bleu']:.2f}  (baseline: {BASELINE['bleu']}, target: >={TARGETS['bleu']})")
    print(f"  ROUGE-L:      {overall_best['rouge_l']:.2f}  (baseline: {BASELINE['rouge_l']}, target: >={TARGETS['rouge_l']})")
    print(f"  BERTScore F1: {overall_best['bertscore_f1']:.2f}")
    print(f"")
    
    bleu_met = overall_best.get("bleu_target_met", False)
    rouge_met = overall_best.get("rouge_target_met", False)
    
    if bleu_met and rouge_met:
        print(f"  ==> GO: 두 목표 모두 달성. 배포를 진행합니다.")
    elif overall_best["bleu"] > 25 or overall_best["rouge_l"] > 35:
        print(f"  ==> PARTIAL GO: 유의미한 개선. 추가 실험 또는 데이터 증강 고려.")
    else:
        print(f"  ==> NO-GO: 목표 미달. 데이터 품질, 증강, 또는 모델 변경 필요.")

In [ ]:
# ============================================================
# 11-2. 최적 모델 재학습 및 저장
# ============================================================

if overall_best and "error" not in overall_best:
    best_cfg = overall_best["config"]
    best_exp_id = overall_best["exp_id"]
    
    print(f"최적 모델 ({best_exp_id}) 로드 중...")
    
    # 최적 실험의 체크포인트 경로
    best_checkpoint_dir = os.path.join(OUTPUT_BASE, best_exp_id, "final")
    
    if os.path.exists(best_checkpoint_dir):
        print(f"체크포인트 발견: {best_checkpoint_dir}")
        
        # LoRA 어댑터를 별도로 저장
        adapter_save_dir = os.path.join(OUTPUT_BASE, "best-adapter")
        os.makedirs(adapter_save_dir, exist_ok=True)
        
        # 체크포인트 복사
        import shutil
        if os.path.exists(adapter_save_dir):
            shutil.rmtree(adapter_save_dir)
        shutil.copytree(best_checkpoint_dir, adapter_save_dir)
        print(f"LoRA 어댑터 저장 완료: {adapter_save_dir}")
        
        # 병합 모델 생성 (선택사항 - 메모리 주의)
        MERGE_MODEL = False  # True로 변경하면 병합 모델 생성
        
        if MERGE_MODEL:
            print("\n병합 모델 생성 중... (시간이 소요됩니다)")
            from peft import PeftModel
            
            # 풀 모델 로드 (양자화 없이)
            base_model = AutoModelForCausalLM.from_pretrained(
                BASE_MODEL_ID,
                torch_dtype=torch.bfloat16,
                device_map="auto",
                trust_remote_code=True,
            )
            
            # LoRA 어댑터 병합
            merged_model = PeftModel.from_pretrained(base_model, adapter_save_dir)
            merged_model = merged_model.merge_and_unload()
            
            # 병합 모델 저장
            merged_save_dir = os.path.join(OUTPUT_BASE, "best-merged")
            merged_model.save_pretrained(merged_save_dir)
            
            tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_ID, trust_remote_code=True)
            tokenizer.save_pretrained(merged_save_dir)
            
            print(f"병합 모델 저장 완료: {merged_save_dir}")
            
            del base_model, merged_model
            gc.collect()
            torch.cuda.empty_cache()
    else:
        print(f"[경고] 체크포인트를 찾을 수 없습니다: {best_checkpoint_dir}")
        print("최적 설정으로 재학습이 필요합니다.")
else:
    print("유효한 최적 모델이 없습니다.")

In [ ]:
# ============================================================
# 11-3. HuggingFace Hub 업로드 (선택사항)
# ============================================================

PUSH_TO_HUB = False  # True로 변경하면 HuggingFace Hub에 업로드
HF_REPO_ID = "your-username/exaone-civil-complaint-qlora"  # 수정 필요

if PUSH_TO_HUB and overall_best and "error" not in overall_best:
    from huggingface_hub import HfApi
    
    api = HfApi()
    adapter_save_dir = os.path.join(OUTPUT_BASE, "best-adapter")
    
    if os.path.exists(adapter_save_dir):
        print(f"HuggingFace Hub 업로드 중: {HF_REPO_ID}")
        api.upload_folder(
            folder_path=adapter_save_dir,
            repo_id=HF_REPO_ID,
            repo_type="model",
            commit_message=f"Upload best QLoRA adapter ({overall_best['exp_id']}): BLEU={overall_best['bleu']:.2f}, ROUGE-L={overall_best['rouge_l']:.2f}",
        )
        print(f"업로드 완료: https://huggingface.co/{HF_REPO_ID}")
    else:
        print(f"어댑터 디렉토리를 찾을 수 없습니다: {adapter_save_dir}")
else:
    print("HuggingFace Hub 업로드를 건너뜁니다. (PUSH_TO_HUB=False)")

In [ ]:
# ============================================================
# 11-4. 최종 결과 JSON 저장 및 W&B 요약
# ============================================================

# 최종 결과 JSON
final_results = {
    "experiment": "QLoRA Hyperparameter Optimization",
    "timestamp": datetime.now().isoformat(),
    "base_model": BASE_MODEL_ID,
    "dataset": {
        "train": TRAIN_PATH,
        "val": VAL_PATH,
        "test": TEST_PATH,
    },
    "baseline": BASELINE,
    "targets": TARGETS,
    "round1_results": round1_results,
    "round2_results": round2_results,
    "round3_results": round3_results,
    "all_results": ALL_RESULTS,
    "best_config": overall_best["config"] if overall_best else None,
    "best_metrics": {
        "bleu": overall_best["bleu"] if overall_best else None,
        "rouge_l": overall_best["rouge_l"] if overall_best else None,
        "bertscore_f1": overall_best["bertscore_f1"] if overall_best else None,
    } if overall_best else None,
}

results_path = os.path.join(RESULTS_DIR, "hparam_optimization_final.json")
with open(results_path, "w", encoding="utf-8") as f:
    json.dump(final_results, f, indent=2, ensure_ascii=False, default=str)

print(f"최종 결과 저장: {results_path}")

# W&B 최종 요약 run
summary_run = wandb.init(
    project=WANDB_PROJECT,
    name="SUMMARY-all-rounds",
    tags=["summary", "final"],
    reinit=True,
)

# 모든 실험 결과를 요약 테이블로
summary_table = wandb.Table(
    columns=["exp_id", "round", "lora_r", "lr", "epochs", "bleu", "rouge_l", "bertscore_f1", "status"]
)

for r in ALL_RESULTS:
    if "bleu" not in r:
        continue
    cfg = r["config"]
    exp_id = r["exp_id"]
    round_name = exp_id.split("-")[0]  # R1, R2, R3
    
    bleu_ok = r.get("bleu_target_met", False)
    rouge_ok = r.get("rouge_target_met", False)
    status = "BOTH" if (bleu_ok and rouge_ok) else "PARTIAL" if (bleu_ok or rouge_ok) else "BELOW"
    
    summary_table.add_data(
        exp_id,
        round_name,
        cfg.get("lora_r", "-"),
        cfg.get("lr", "-"),
        cfg.get("epochs", "-"),
        r["bleu"],
        r["rouge_l"],
        r["bertscore_f1"],
        status,
    )

wandb.log({"summary_table": summary_table})

if overall_best:
    wandb.log({
        "best/exp_id": overall_best["exp_id"],
        "best/bleu": overall_best["bleu"],
        "best/rouge_l": overall_best["rouge_l"],
        "best/bertscore_f1": overall_best["bertscore_f1"],
        "best/bleu_improvement": overall_best["bleu"] - BASELINE["bleu"],
        "best/rouge_improvement": overall_best["rouge_l"] - BASELINE["rouge_l"],
    })

wandb.finish()

print("\n" + "="*65)
print("  모든 실험 완료!")
print(f"  결과 파일: {results_path}")
print(f"  W&B 대시보드: https://wandb.ai/project/{WANDB_PROJECT}")
print("="*65)